# Actuarial Pricing Engine Notebook
This notebook implements the foundational math for life insurance pricing using standard actuarial discounting and the loaded mortality table (`data/morality_table.csv`), loading parameters from `config.py`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../')
import config

# Load mortality table (note: resolving to morality_table.csv as committed)
df = pd.read_csv('../data/morality_table.csv')
print(f"Mortality table loaded: {df.shape[0]} ages.")
df.head()

### Actuarial Formulations

1. **Discounting Factor**:
   $$v = \frac{1}{1 + i}$$

2. **Survival Probabilities** (${}_t p_x$):
   $${}_t p_x = \prod_{s=0}^{t-1} (1 - q_{x+s})$$

3. **Net Single Premium (NSP) for Term Life Insurance** ($A^1_{x:\overline{n}|}$):
   $$A^1_{x:\overline{n}|} = \sum_{t=0}^{n-1} v^{t+1} \cdot {}_t p_x \cdot q_{x+t}$$

4. **Temporary Annuity Due** ($\ddot{a}_{x:\overline{n}|}$):
   $$\ddot{a}_{x:\overline{n}|} = \sum_{t=0}^{n-1} v^t \cdot {}_t p_x$$

5. **Level Annual Premium** ($P$):
   $$P = \frac{\text{NSP} \cdot \text{Sum Assured}}{\ddot{a}_{x:\overline{n}|}}$$

In [ ]:
def calculate_level_premium(age, gender, term, sum_assured, interest_rate):
    i = interest_rate / 100.0
    v = 1 / (1 + i)
    base_rates = df['qx'].values
    
    # Apply actuarial gender factor loaded from config
    gender_factor = config.GENDER_FACTORS[gender]
    rates = base_rates * gender_factor
    
    # Survival array
    tpx = np.zeros(len(df) - age)
    tpx[0] = 1.0
    for t in range(1, len(tpx)):
        tpx[t] = tpx[t-1] * (1.0 - rates[age + t - 1])
        
    n = min(term, len(tpx) - 1)
    
    # NSP
    nsp = sum((v ** (t + 1)) * tpx[t] * rates[age + t] for t in range(n))
    
    # Annuity due
    a_due = sum((v ** t) * tpx[t] for t in range(n))
    
    net_single_premium = nsp * sum_assured
    level_annual_premium = net_single_premium / a_due if a_due > 0 else net_single_premium
    
    return net_single_premium, level_annual_premium

nsp, lap = calculate_level_premium(config.DEFAULT_AGE, 'Male', config.DEFAULT_TERM, config.SUM_ASSURED, config.INTEREST_RATE * 100)
print(f"{config.DEFAULT_AGE} M, {config.SUM_ASSURED} Sum Assured, {config.DEFAULT_TERM}-Year Term, {config.INTEREST_RATE*100}% Interest:")
print(f"  Net Single Premium: ${nsp:,.2f}")
print(f"  Level Annual Premium: ${lap:,.2f}")

In [ ]:
# Plot mortality table rates
plt.figure(figsize=(10, 6))
plt.plot(df['Age'], df['qx'], label='Base qx', color='blue')
plt.plot(df['Age'], df['qx'] * config.GENDER_FACTORS['Male'], label='Male qx (Adjusted)', color='orange', linestyle='--')
plt.plot(df['Age'], df['qx'] * config.GENDER_FACTORS['Female'], label='Female qx (Adjusted)', color='red', linestyle='--')
plt.yscale('log')
plt.title('Mortality Rate (qx) in Log Scale by Age')
plt.xlabel('Age')
plt.ylabel('qx (Log Scale)')
plt.grid(True, which="both", ls="--")
plt.legend()
plt.show()